# Preanalysis

## Data Importing

In [46]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [47]:
price_history_electricity = pd.read_csv('data/Price History_Electricity.csv', sep=';', decimal=',')
price_history_gas = pd.read_csv('data/Price History_Gas.csv')
price_history_carbon = pd.read_csv('data/Price History_Carbon.csv')
price_history_coal = pd.read_csv('data/Price History_Coal.csv')
price_history_EURUSD = pd.read_csv('data/Price History_EUR:USD.csv')
price_history_EURRUB = pd.read_csv('data/Price History_EUR:RUB.csv')
price_history_EURDKR = pd.read_csv('data/Price History_EUR:DKR.csv')
price_history_EURNKR = pd.read_csv('data/Price History_EUR:NKR.csv')

/var/folders/3t/z_28kb7d61s_qsrgn944bmzh0000gn/T/ipykernel_1592/877982077.py:1: DtypeWarning: Columns (0: Frankreich [€/MWh] Berechnete Auflösungen, 1: Slowenien [€/MWh] Berechnete Auflösungen, 2: Ungarn [€/MWh] Berechnete Auflösungen) have mixed types. Specify dtype option on import or set low_memory=False.
  price_history_electricity = pd.read_csv('data/Price History_Electricity.csv', sep=';', decimal=',')


In [48]:
# averaging price of electricity
col_de = 'Deutschland/Luxemburg [€/MWh] Berechnete Auflösungen'
price_history_electricity['ts'] = pd.to_datetime(price_history_electricity['Datum von'], dayfirst=True)

price_history_electricity[col_de] = pd.to_numeric(price_history_electricity[col_de], errors='coerce')
price_history_electricity = (
    price_history_electricity.assign(date=lambda d: d['ts'].dt.normalize())
    .groupby('date', as_index=False)[col_de]
    .mean()
    .rename(columns={col_de: 'avg_price'})
)

price_history_electricity.head()

,date,avg_price
0,2021-01-01,48.398333
1,2021-01-02,50.562500
2,2021-01-03,38.622500
3,2021-01-04,48.017917
4,2021-01-05,55.337083


In [49]:
# handle commodities
def _handle_commodities(df):
    df['Exchange Date'] = pd.to_datetime(df['Exchange Date'], dayfirst=True)
    df['date'] = df['Exchange Date']
    df['close'] = df['Close']
    return df[['date', 'close']]

# handle forex
def _handle_forex(df):
    df['Exchange Date'] = pd.to_datetime(df['Exchange Date'], dayfirst=True)
    df['date'] = df['Exchange Date']
    df['mid'] = (df['Bid'] + df['Ask']) / 2
    return df[['date', 'mid']]

In [50]:
price_history_gas = _handle_commodities(price_history_gas)
price_history_carbon = _handle_commodities(price_history_carbon)
price_history_coal = _handle_commodities(price_history_coal)
price_history_EURUSD = _handle_forex(price_history_EURUSD)
price_history_EURRUB = _handle_forex(price_history_EURRUB)
price_history_EURDKR = _handle_forex(price_history_EURDKR)
price_history_EURNKR = _handle_forex(price_history_EURNKR)

In [51]:
print(f'Electricity:\n{price_history_electricity.isna().sum()}')
print(f'Gas:\n{price_history_gas.isna().sum()}')
print(f'Carbon:\n{price_history_carbon.isna().sum()}')
print(f'Coal:\n{price_history_coal.isna().sum()}')
print(f'EURUSD:\n{price_history_EURUSD.isna().sum()}')
print(f'EURRUB:\n{price_history_EURRUB.isna().sum()}')
print(f'EURDKR:\n{price_history_EURDKR.isna().sum()}')
print(f'EURNKR:\n{price_history_EURNKR.isna().sum()}')

Electricity:
date         0
avg_price    0
dtype: int64
Gas:
date     0
close    0
dtype: int64
Carbon:
date     0
close    0
dtype: int64
Coal:
date     0
close    0
dtype: int64
EURUSD:
date    0
mid     0
dtype: int64
EURRUB:
date    0
mid     0
dtype: int64
EURDKR:
date    0
mid     0
dtype: int64
EURNKR:
date    0
mid     0
dtype: int64


In [52]:
def _forward_fill_missing_calendar_days(df, date_col="date", end_date=None):
    out = df.copy()
    out[date_col] = pd.to_datetime(out[date_col])
    out = out.sort_values(date_col).set_index(date_col)

    start = out.index.min()
    end = pd.to_datetime(end_date) if end_date is not None else out.index.max()

    full_idx = pd.date_range(start, end, freq="D")
    out = out.reindex(full_idx).ffill()

    out.index.name = date_col
    return out.reset_index()

In [53]:
price_history_gas = _forward_fill_missing_calendar_days(price_history_gas, end_date=price_history_electricity['date'].max())
price_history_carbon = _forward_fill_missing_calendar_days(price_history_carbon, end_date=price_history_electricity['date'].max())
price_history_coal = _forward_fill_missing_calendar_days(price_history_coal, end_date=price_history_electricity['date'].max())
price_history_EURUSD = _forward_fill_missing_calendar_days(price_history_EURUSD, end_date=price_history_electricity['date'].max())
price_history_EURRUB = _forward_fill_missing_calendar_days(price_history_EURRUB, end_date=price_history_electricity['date'].max())
price_history_EURDKR = _forward_fill_missing_calendar_days(price_history_EURDKR, end_date=price_history_electricity['date'].max())
price_history_EURNKR = _forward_fill_missing_calendar_days(price_history_EURNKR, end_date=price_history_electricity['date'].max())